In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# -----------------------------
# EDIT THESE
# -----------------------------

MODEL_SUMMARY_PATH = Path("/home/akapociu/ift/interactiondynamics/results/test666/summary.jsonl")

BASELINE_SUMMARY_PATH = Path(
    "/home/akapociu/ift/interactiondynamics/results/whole_bin_dummy_baselines/dummy_baselines_summary.jsonl"
)

# Option A: load a model row from MODEL_SUMMARY_PATH by line number
MODEL_ROW_INDEX = 0

# Option B: paste one raw JSONL line here instead.
# Leave as None to use MODEL_ROW_INDEX.
MODEL_ROW_JSON = None

# Which split to compare
SPLIT = "test"

# Metrics to compare
METRICS = [
    "micro_pr_auc",
    "micro_f1",
    "micro_jaccard",
    "macro_pr_auc",
]

In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


if MODEL_ROW_JSON is not None:
    model_row = json.loads(MODEL_ROW_JSON)
else:
    model_rows = load_jsonl(MODEL_SUMMARY_PATH)
    print(f"Loaded {len(model_rows)} model rows from {MODEL_SUMMARY_PATH}")
    model_row = model_rows[MODEL_ROW_INDEX]

model_row.keys()

In [ ]:
print("Top-level keys:")
print(model_row.keys())

print("\nDataset:")
print(model_row.get("dataset"))

print("\nRun:")
print(model_row.get("run") or model_row.get("name") or model_row.get("model"))

print("\nBest snapshot keys:")
best_snapshot = model_row.get("best_snapshot", {})
print(best_snapshot.keys() if isinstance(best_snapshot, dict) else type(best_snapshot))

In [ ]:
def get_nested_metrics(row, split="test", snapshot_key="best_snapshot"):
    """
    Try to extract split metrics from a model summary row.

    Expected common format:
        row["best_snapshot"]["test"]["micro_pr_auc"]

    Also supports flatter formats like:
        row["best_snapshot"]["test_micro_pr_auc"]
        row["test"]["micro_pr_auc"]
        row["micro_pr_auc"]
    """
    # 1. best_snapshot -> split -> metrics
    snap = row.get(snapshot_key, {})
    if isinstance(snap, dict):
        if split in snap and isinstance(snap[split], dict):
            return snap[split]

        # 2. best_snapshot has prefixed keys like test_micro_pr_auc
        prefixed = {}
        prefix = f"{split}_"
        for k, v in snap.items():
            if isinstance(k, str) and k.startswith(prefix):
                prefixed[k[len(prefix):]] = v
        if prefixed:
            return prefixed

    # 3. top-level split dict
    if split in row and isinstance(row[split], dict):
        return row[split]

    # 4. top-level prefixed keys
    prefixed = {}
    prefix = f"{split}_"
    for k, v in row.items():
        if isinstance(k, str) and k.startswith(prefix):
            prefixed[k[len(prefix):]] = v
    if prefixed:
        return prefixed

    # 5. top-level unprefixed metric keys
    return row


def model_row_to_compare_record(row, split="test", metrics=METRICS):
    m = get_nested_metrics(row, split=split)

    method = (
        row.get("run")
        or row.get("name")
        or row.get("model")
        or row.get("baseline")
        or "neural_model"
    )

    dataset = row.get("dataset", "unknown_dataset")

    rec = {
        "dataset": dataset,
        "method": method,
        "kind": "neural_model",
        "split": split,
    }

    for metric in metrics:
        rec[metric] = m.get(metric, float("nan"))

    # Useful extras if present
    for extra in [
        "edge_density",
        "micro_edge_density",
        "macro_edge_density",
        "selected_threshold",
        "threshold",
        "best_epoch",
        "seed",
    ]:
        if extra in m:
            rec[extra] = m[extra]
        elif extra in row:
            rec[extra] = row[extra]

    # Compatibility fallback:
    # If you have not added micro/macro names yet, old pr_auc is macro-ish.
    if pd.isna(rec.get("macro_pr_auc")) and "pr_auc" in m:
        rec["macro_pr_auc"] = m["pr_auc"]

    return rec


model_rec = model_row_to_compare_record(model_row, split=SPLIT)
model_rec

In [ ]:
baseline_df = pd.read_json(BASELINE_SUMMARY_PATH, lines=True)

print("Baseline columns:")
print(baseline_df.columns.tolist())

print("\nUnique baseline datasets:")
for d in baseline_df["dataset"].unique():
    print("-", d)

baseline_df.head()

In [ ]:
model_dataset = str(model_rec["dataset"])

# First try exact match.
candidate_baselines = baseline_df[
    (baseline_df["split"] == SPLIT) &
    (baseline_df["dataset"] == model_dataset)
].copy()

# If exact match fails, try fuzzy contains.
if candidate_baselines.empty:
    fuzzy = baseline_df[
        (baseline_df["split"] == SPLIT) &
        (
            baseline_df["dataset"].astype(str).str.contains(model_dataset, regex=False)
            | pd.Series([model_dataset]).iloc[0].__contains__(baseline_df["dataset"].astype(str).iloc[0])
        )
    ].copy()

# Safer fuzzy matching:
if candidate_baselines.empty:
    possible = baseline_df[baseline_df["split"] == SPLIT]["dataset"].unique().tolist()
    print("No exact dataset match found.")
    print("\nModel dataset:")
    print(model_dataset)
    print("\nAvailable baseline datasets:")
    for i, d in enumerate(possible):
        print(f"{i}: {d}")

    BASELINE_DATASET_INDEX = 0  # <-- edit this after looking at printed list
    baseline_dataset = possible[BASELINE_DATASET_INDEX]
    candidate_baselines = baseline_df[
        (baseline_df["split"] == SPLIT) &
        (baseline_df["dataset"] == baseline_dataset)
    ].copy()
else:
    baseline_dataset = model_dataset

print("Using baseline dataset:")
print(candidate_baselines["dataset"].iloc[0])

candidate_baselines[["dataset", "baseline", "split"] + [m for m in METRICS if m in candidate_baselines.columns]]

In [ ]:
def baseline_rows_to_compare_records(df, metrics=METRICS):
    records = []

    for _, row in df.iterrows():
        rec = {
            "dataset": row["dataset"],
            "method": row["baseline"],
            "kind": "dummy_baseline",
            "split": row["split"],
        }

        for metric in metrics:
            rec[metric] = row.get(metric, float("nan"))

        for extra in [
            "micro_edge_density",
            "macro_edge_density",
            "train_edge_density",
            "selected_threshold",
            "threshold",
        ]:
            if extra in row:
                rec[extra] = row[extra]

        records.append(rec)

    return records


baseline_records = baseline_rows_to_compare_records(candidate_baselines)
combined = pd.DataFrame(baseline_records + [model_rec])

# Put neural model near top or sort by a metric.
combined_sorted = combined.sort_values("micro_pr_auc", ascending=False)

combined_sorted[
    ["kind", "method"] + METRICS + [
        c for c in ["micro_edge_density", "train_edge_density", "selected_threshold"] 
        if c in combined_sorted.columns
    ]
]

In [ ]:
metric = "micro_pr_auc"

plot_df = combined.sort_values(metric, ascending=False).copy()

plt.figure(figsize=(12, 5))
plt.bar(plot_df["method"], plot_df[metric])
plt.xticks(rotation=60, ha="right")
plt.ylabel(metric)
plt.title(f"{metric} comparison — {SPLIT}")
plt.tight_layout()
plt.show()

In [ ]:
for metric in METRICS:
    if metric not in combined.columns:
        print(f"Skipping missing metric: {metric}")
        continue

    plot_df = combined.sort_values(metric, ascending=False).copy()

    plt.figure(figsize=(12, 5))
    plt.bar(plot_df["method"], plot_df[metric])
    plt.xticks(rotation=60, ha="right")
    plt.ylabel(metric)
    plt.title(f"{metric} comparison — {SPLIT}")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_long = combined.melt(
    id_vars=["method", "kind"],
    value_vars=[m for m in METRICS if m in combined.columns],
    var_name="metric",
    value_name="value",
)

# Keep methods ordered by micro_pr_auc if present.
method_order = (
    combined.sort_values("micro_pr_auc", ascending=False)["method"].tolist()
    if "micro_pr_auc" in combined.columns
    else combined["method"].tolist()
)

plot_long["method"] = pd.Categorical(
    plot_long["method"],
    categories=method_order,
    ordered=True,
)

pivot = plot_long.pivot_table(
    index="method",
    columns="metric",
    values="value",
    aggfunc="first",
)

ax = pivot.plot(kind="bar", figsize=(14, 6))
ax.set_ylabel("score")
ax.set_title(f"Dummy baselines vs neural model — {SPLIT}")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
density_col = None

for c in ["micro_edge_density", "macro_edge_density", "train_edge_density"]:
    if c in combined.columns and combined[c].notna().any():
        density_col = c
        break

if density_col is None:
    print("No edge density column found.")
else:
    density_df = combined.copy()
    density_df = density_df.sort_values("micro_pr_auc", ascending=False)

    fig, ax1 = plt.subplots(figsize=(12, 5))

    ax1.bar(density_df["method"], density_df["micro_pr_auc"])
    ax1.set_ylabel("micro_pr_auc")
    ax1.set_xticklabels(density_df["method"], rotation=60, ha="right")

    ax2 = ax1.twinx()
    ax2.plot(density_df["method"], density_df[density_col], marker="o")
    ax2.set_ylabel(density_col)

    plt.title(f"micro_pr_auc with edge density overlay — {SPLIT}")
    plt.tight_layout()
    plt.show()

In [ ]:
main_metric = "micro_pr_auc"

ranked = combined.sort_values(main_metric, ascending=False).reset_index(drop=True)

print(f"Ranking by {main_metric}:")
display(ranked[["kind", "method", main_metric] + [m for m in METRICS if m != main_metric]])

neural_score = combined.loc[combined["kind"] == "neural_model", main_metric].iloc[0]

baseline_only = combined[combined["kind"] == "dummy_baseline"].copy()
best_baseline_row = baseline_only.sort_values(main_metric, ascending=False).iloc[0]

print("\nBest dummy baseline:")
print(best_baseline_row["method"], best_baseline_row[main_metric])

print("\nNeural model:")
print(model_rec["method"], neural_score)

gap = neural_score - best_baseline_row[main_metric]
print("\nGap neural - best dummy baseline:")
print(gap)

if gap > 0:
    print("Neural model beats the strongest dummy baseline on this metric.")
else:
    print("Dummy baseline beats or ties the neural model on this metric. That is important.")

In [ ]:
SCARY_BASELINES = [
    "persistence",
    "last_k_union",
    "edge_frequency",
    "degree_product",
]

scary_compare = combined[
    (combined["kind"] == "neural_model") |
    (combined["method"].isin(SCARY_BASELINES))
].copy()

scary_compare = scary_compare.sort_values("micro_pr_auc", ascending=False)

display(scary_compare[["kind", "method"] + METRICS])

plt.figure(figsize=(10, 5))
plt.bar(scary_compare["method"], scary_compare["micro_pr_auc"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("micro_pr_auc")
plt.title("Neural model vs scary dumb baselines")
plt.tight_layout()
plt.show()